# 02 — PGD Adversarial Attack Evaluation

Runs projected gradient descent (PGD, L∞) on the fixed MNIST evaluation subset
and records per-sample clean/adversarial predictions in a CSV.

**Thesis context — WP1:** Measures empirical robustness of the trained model.
The fixed evaluation split lives at `assets/splits/mnist_eval_1000.json` (1000 samples,
seed=1234) — its first 100 indices match the legacy `mnist_eval_100.json` exactly,
so this notebook's results are a strict superset of the original baseline.

**Run twice to compare models:**
1. With `CKPT_PATH = "runs/mlp_mnist/model.pt"` (standard training, from `01_train_mnist.ipynb`).
2. With `CKPT_PATH = "runs/mlp_ibp_trained/model.pt"` (IBP-robust training, from `08_train_ibp_mlp.ipynb`).

Each run writes `results/pgd_<run_name>_eps<eps>.csv`.

**Prerequisites:** Either run `01_train_mnist.ipynb` / `08_train_ibp_mlp.ipynb` first,
or mount Google Drive where the checkpoint already exists and set `CKPT_PATH` below.

In [1]:
# ── Colab bootstrap ─────────────────────────────────────────────────────────────
# Mounts Drive (Colab) or noop (local), cd's into the project folder, and
# self-generates `assets/splits/mnist_eval_1000.json` if it's not present.
# colab_bootstrap_v1
import os
from pathlib import Path

try:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/drive/MyDrive/thesis-formal-verification")
    PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
    os.chdir(PROJECT_ROOT)
    print(f"[colab] working dir: {PROJECT_ROOT}")
except ImportError:
    print(f"[local] working dir: {Path.cwd()}")

# Self-generate the 1000-sample evaluation split if missing
import json, random
split_100  = Path("assets/splits/mnist_eval_100.json")
split_1000 = Path("assets/splits/mnist_eval_1000.json")
if not split_1000.exists():
    split_100.parent.mkdir(parents=True, exist_ok=True)
    seed = 1234
    if split_100.exists():
        idx100 = json.loads(split_100.read_text(encoding="utf-8"))["indices"]
    else:
        idx100 = random.Random(seed).sample(range(10_000), 100)
        split_100.write_text(json.dumps({"seed": seed, "indices": idx100}, indent=2) + "\n", encoding="utf-8")
    remaining = [i for i in range(10_000) if i not in set(idx100)]
    extra = random.Random(seed).sample(remaining, 900)
    indices_1000 = idx100 + extra
    assert indices_1000[:100] == idx100
    split_1000.write_text(json.dumps({"seed": seed, "indices": indices_1000}, indent=2) + "\n", encoding="utf-8")
    print(f"[bootstrap] wrote {split_1000} ({len(indices_1000)} indices)")
else:
    print(f"[bootstrap] OK {split_1000}  ({len(json.loads(split_1000.read_text(encoding='utf-8'))['indices'])} indices)")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[colab] working dir: /content/drive/MyDrive/thesis-formal-verification
[bootstrap] OK assets/splits/mnist_eval_1000.json  (1000 indices)


In [2]:
!pip install -q torch torchvision numpy pandas pyyaml tqdm ortools

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.8/29.8 MB 70.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.4/323.4 kB 31.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 6.33.6 which is incompatible.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 6.33.6 which is incompatible.


## 1 — Library code

In [3]:
from __future__ import annotations

import json
import os
import random
import time
from dataclasses import asdict, dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms


# ── Reproducibility ────────────────────────────────────────────────────────────
def set_seed(seed: int, deterministic: bool = True) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        torch.use_deterministic_algorithms(True, warn_only=True)


# ── Data ───────────────────────────────────────────────────────────────────────
def get_mnist_datasets(data_dir: str | Path = "data"):
    data_dir = Path(data_dir)
    tfm = transforms.ToTensor()
    train_ds = datasets.MNIST(root=str(data_dir), train=True,  download=True, transform=tfm)
    test_ds  = datasets.MNIST(root=str(data_dir), train=False, download=True, transform=tfm)
    return train_ds, test_ds


# ── Evaluation split ───────────────────────────────────────────────────────────
@dataclass(frozen=True)
class Split:
    seed: int
    indices: list[int]


def load_split(path: str | Path) -> Split:
    obj = json.loads(Path(path).read_text(encoding="utf-8"))
    return Split(seed=int(obj["seed"]), indices=[int(i) for i in obj["indices"]])


def save_split(path: str | Path, split: Split) -> None:
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    Path(path).write_text(
        json.dumps({"seed": split.seed, "indices": split.indices}, indent=2) + "\n",
        encoding="utf-8",
    )


def ensure_mnist_eval_split(path: str | Path, seed: int = 1234, n: int = 100) -> Split:
    p = Path(path)
    if p.exists():
        return load_split(p)
    indices = random.Random(seed).sample(range(10_000), n)
    split = Split(seed=seed, indices=indices)
    save_split(p, split)
    return split


# ── Models ─────────────────────────────────────────────────────────────────────
class MnistMlp(nn.Module):
    def __init__(self, in_dim=784, h1=128, h2=64, num_classes=10):
        super().__init__()
        self.fc1 = nn.Linear(int(in_dim), int(h1))
        self.fc2 = nn.Linear(int(h1), int(h2))
        self.fc3 = nn.Linear(int(h2), int(num_classes))
        self.relu = nn.ReLU()

    def forward(self, x):
        if x.ndim == 4:
            x = x.view(x.shape[0], -1)
        return self.fc3(self.relu(self.fc2(self.relu(self.fc1(x)))))

    def linear_layers(self):
        return [self.fc1, self.fc2, self.fc3]


class CnnSmall(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Linear(32*7*7, 64), nn.ReLU(), nn.Linear(64, int(num_classes)),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


# ── Checkpoint I/O ─────────────────────────────────────────────────────────────
@dataclass(frozen=True)
class CheckpointMeta:
    model_type: str
    model_kwargs: dict
    run_name: str


def build_model(model_type, model_kwargs):
    if model_type == "mlp":       return MnistMlp(**model_kwargs)
    if model_type == "cnn_small": return CnnSmall(**model_kwargs)
    raise ValueError(f"Unknown model_type={model_type!r}")


def load_checkpoint(path, map_location="cpu"):
    payload = torch.load(str(path), map_location=map_location)
    raw = payload["meta"]
    meta = CheckpointMeta(
        model_type=str(raw["model_type"]),
        model_kwargs=dict(raw.get("model_kwargs", {})),
        run_name=str(raw.get("run_name", "run")),
    )
    model = build_model(meta.model_type, meta.model_kwargs)
    model.load_state_dict(payload["model_state_dict"])
    return model, meta, payload.get("metrics", {})


# ── PGD attack ─────────────────────────────────────────────────────────────────
@torch.no_grad()
def _clamp_linf(x, x0, eps):
    lo = (x0 - float(eps)).clamp(0.0, 1.0)
    hi = (x0 + float(eps)).clamp(0.0, 1.0)
    return x.clamp(lo, hi)


def pgd_linf(model, x0, y, eps, steps, step_size, random_start=False):
    model.eval()
    x0 = x0.detach()
    x  = x0.clone()
    if random_start:
        x = _clamp_linf(x + (2 * torch.rand_like(x) - 1) * float(eps), x0, eps)
    for _ in range(int(steps)):
        x.requires_grad_(True)
        loss = F.cross_entropy(model(x), y, reduction="sum")
        grad = torch.autograd.grad(loss, x)[0]
        with torch.no_grad():
            x = _clamp_linf(x + float(step_size) * grad.sign(), x0, eps)
        x = x.detach()
    return x


print("Library code loaded")

Library code loaded


## 2 — Configuration

Set `CKPT_PATH` to point to a trained model checkpoint.
If running after `01_train_mnist.ipynb` in the same session, the default should work.

In [9]:
# ── Configuration ──────────────────────────────────────────────────────────────
# Pick ONE of these two checkpoints (run the cell once per checkpoint):
# CKPT_PATH    = "runs/mlp_mnist/model.pt"           # standard-trained MLP (from 01)
CKPT_PATH    = "runs/mlp_ibp_trained/model.pt"   # IBP-trained MLP (from 08)

SUBSET_PATH  = "assets/splits/mnist_eval_1000.json"  # 1000 samples (1st 100 = legacy split)
DATA_DIR     = "data"
EPS          = 0.03     # L-inf perturbation radius
STEPS        = 40       # PGD iterations
STEP_SIZE    = 0.01     # step size (alpha)
RANDOM_START = True     # random initialisation
BATCH_SIZE   = 128      # bumped from 64 — 1000 samples * batch=128 ~ 8 batches
DEVICE       = "cuda" if torch.cuda.is_available() else "cpu"
# ──────────────────────────────────────────────────────────────────────────────

print(f"Device: {DEVICE} | eps={EPS} | steps={STEPS} | subset={SUBSET_PATH}")

Device: cuda | eps=0.03 | steps=40 | subset=assets/splits/mnist_eval_1000.json


## 3 — Ensure evaluation split exists

In [10]:
from pathlib import Path

if not Path(SUBSET_PATH).exists():
    raise FileNotFoundError(
        f"Eval split not found at {SUBSET_PATH!r}. "
        "Pull the latest repo (the 1000-sample split is committed at "
        "`assets/splits/mnist_eval_1000.json`)."
    )
split = load_split(SUBSET_PATH)
print(f"Evaluation split: {len(split.indices)} samples (seed={split.seed})")

Evaluation split: 1000 samples (seed=1234)


## 4 — Load model

In [11]:
model, meta, _ = load_checkpoint(CKPT_PATH, map_location=DEVICE)
model.to(DEVICE).eval()
print(f"Loaded model: {meta.model_type!r} (run={meta.run_name!r})")

Loaded model: 'mlp' (run='mlp_ibp_trained')


## 5 — Run PGD evaluation

In [12]:
@torch.no_grad()
def batch_metrics(model, x, y):
    logits = model(x)
    loss   = F.cross_entropy(logits, y, reduction="none")
    return logits.argmax(dim=1), loss


_, test_ds = get_mnist_datasets(DATA_DIR)
sub_ds     = Subset(test_ds, split.indices)
loader     = DataLoader(sub_ds, batch_size=int(BATCH_SIZE), shuffle=False, num_workers=0)

rows  = []
start = time.time()

for batch_idx, (x0, y) in enumerate(loader):
    x0, y = x0.to(DEVICE), y.to(DEVICE)
    clean_pred, clean_loss = batch_metrics(model, x0, y)
    x_adv = pgd_linf(
        model, x0=x0, y=y, eps=float(EPS),
        steps=int(STEPS), step_size=float(STEP_SIZE), random_start=bool(RANDOM_START),
    )
    adv_pred, adv_loss = batch_metrics(model, x_adv, y)

    base = batch_idx * int(BATCH_SIZE)
    for i in range(y.shape[0]):
        rows.append({
            "index":        int(split.indices[base + i]),
            "y":            int(y[i].item()),
            "clean_pred":   int(clean_pred[i].item()),
            "adv_pred":     int(adv_pred[i].item()),
            "success":      int(adv_pred[i].item()) != int(y[i].item()),
            "clean_loss":   float(clean_loss[i].item()),
            "adv_loss":     float(adv_loss[i].item()),
            "eps":          float(EPS),
            "steps":        int(STEPS),
            "step_size":    float(STEP_SIZE),
            "random_start": bool(RANDOM_START),
            "run_name":     meta.run_name,
        })

elapsed = time.time() - start
df = pd.DataFrame(rows)

Path("results").mkdir(parents=True, exist_ok=True)
out_path = f"results/pgd_{meta.run_name}_eps{EPS:.4f}.csv"
df.to_csv(out_path, index=False)

asr = df["success"].mean()
print(f"Attack success rate (ASR): {asr:.4f} ({df['success'].sum()}/{len(df)})")
print(f"Clean accuracy on subset: {(df['clean_pred'] == df['y']).mean():.4f}")
print(f"Results saved to: {out_path} ({len(df)} rows) in {elapsed:.2f}s")

Attack success rate (ASR): 0.0410 (41/1000)
Clean accuracy on subset: 0.9800
Results saved to: results/pgd_mlp_ibp_trained_eps0.0300.csv (1000 rows) in 0.50s


## 6 — Summary

In [13]:
print(df.groupby("y")[["success", "clean_loss", "adv_loss"]].mean().round(4).to_string())
df.head(10)

   success  clean_loss  adv_loss
y                               
0   0.0000      0.0007    0.0049
1   0.0086      0.0039    0.0282
2   0.0541      0.0417    0.1175
3   0.0490      0.0538    0.1171
4   0.0213      0.0202    0.0651
5   0.0543      0.0378    0.1383
6   0.0435      0.0671    0.1716
7   0.0233      0.0148    0.0863
8   0.0804      0.1419    0.2948
9   0.0722      0.0806    0.2012


,index,y,clean_pred,adv_pred,success,clean_loss,adv_loss,eps,steps,step_size,random_start,run_name
0,7220,8,8,8,False,0.013341,0.096288,0.03,40,0.01,True,mlp_ibp_trained
1,1914,8,8,8,False,0.000081,0.000400,0.03,40,0.01,True,mlp_ibp_trained
2,122,7,7,7,False,0.000100,0.001302,0.03,40,0.01,True,mlp_ibp_trained
3,1485,2,2,2,False,0.000822,0.002306,0.03,40,0.01,True,mlp_ibp_trained
4,9539,9,9,9,False,0.000280,0.001353,0.03,40,0.01,True,mlp_ibp_trained
5,572,8,8,8,False,0.000527,0.001354,0.03,40,0.01,True,mlp_ibp_trained
6,1375,2,2,2,False,0.000230,0.000602,0.03,40,0.01,True,mlp_ibp_trained
7,1612,3,3,3,False,0.000004,0.000169,0.03,40,0.01,True,mlp_ibp_trained
8,5810,6,6,6,False,0.000019,0.000160,0.03,40,0.01,True,mlp_ibp_trained
9,3879,9,9,9,False,0.000629,0.010091,0.03,40,0.01,True,mlp_ibp_trained
